# Agent-to-Agent Collaboration Platform

A local Python multi-agent workflow demonstrating specialized agents, task dependencies, structured message passing, shared state, review, retries, and auditability. Uses deterministic Python agents and synthetic evidence; no external LLM or API calls.

## 1. Setup

In [ ]:
# If needed: %pip install pandas numpy matplotlib
import os, json
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from collections import defaultdict, deque
import pandas as pd, numpy as np, matplotlib.pyplot as plt


## 2. Collaboration primitives

In [ ]:
@dataclass
class Task:
    task_id:str; title:str; owner:str; dependencies:list=field(default_factory=list)
    status:str="pending"; input_data:dict=field(default_factory=dict); output:dict=field(default_factory=dict); attempts:int=0
@dataclass
class Message:
    message_id:str; sender:str; recipient:str; task_id:str; kind:str; payload:dict; timestamp:str
class MessageBus:
    def __init__(self): self.messages=[]
    def send(self,sender,recipient,task_id,kind,payload):
        m=Message(f"MSG-{len(self.messages)+1:04d}",sender,recipient,task_id,kind,payload,datetime.now(timezone.utc).isoformat())
        self.messages.append(m); return m
class Workspace:
    def __init__(self): self.artifacts={}; self.audit=[]
    def publish(self,key,value,agent):
        self.artifacts[key]=value; self.audit.append({"event":"publish","key":key,"agent":agent,"time":datetime.now(timezone.utc).isoformat()})
bus=MessageBus(); workspace=Workspace()


## 3. Specialized agents

In [ ]:
class Agent:
    def __init__(self,name): self.name=name
    def report(self,tid,status,details): bus.send(self.name,"coordinator",tid,"status",{"status":status,"details":details})
class Researcher(Agent):
    def run(self,t):
        items=t.input_data["evidence"]
        result={"findings":items,"count":len(items)}
        workspace.publish("research",result,self.name); self.report(t.task_id,"completed",{"count":len(items)})
        return result
class Analyst(Agent):
    def run(self,t):
        findings=workspace.artifacts["research"]["findings"]
        trusted=[x for x in findings if x["confidence"]>=.75]
        result={"summary":f"Reviewed {len(findings)} evidence items; {len(trusted)} passed the illustrative confidence threshold.",
                "key_findings":trusted,"caveat":"Confidence values are supplied demo metadata, not calibrated probabilities."}
        workspace.publish("analysis",result,self.name); self.report(t.task_id,"completed",{"reviewed":len(findings)})
        return result
class Writer(Agent):
    def run(self,t):
        a=workspace.artifacts["analysis"]
        bullets="\n".join(f"- {x['claim']} (source: {x['source']})" for x in a["key_findings"])
        draft="# Agent Collaboration Brief\n\n"+a["summary"]+"\n\n## Findings\n"+bullets+"\n\n## Caveat\n"+a["caveat"]
        result={"draft":draft}; workspace.publish("draft",result,self.name)
        self.report(t.task_id,"completed",{"characters":len(draft)}); return result
class Reviewer(Agent):
    def run(self,t):
        d=workspace.artifacts["draft"]["draft"]
        checks={"title":d.startswith("# "),"findings":"## Findings" in d,"caveat":"## Caveat" in d,"nonempty":len(d)>50}
        result={"approved":all(checks.values()),"checks":checks,"feedback":"Ready for human review." if all(checks.values()) else "Revise draft."}
        workspace.publish("review",result,self.name); self.report(t.task_id,"completed",result); return result
agents={"researcher":Researcher("researcher"),"analyst":Analyst("analyst"),"writer":Writer("writer"),"reviewer":Reviewer("reviewer")}
print("Agents:",list(agents))


## 4. Create a dependency-aware workflow

In [ ]:
tasks={
"T1":Task("T1","Collect supplied evidence","researcher",input_data={"evidence":[
 {"claim":"Shared task state can coordinate agent work.","source":"demo-note-A","confidence":.91},
 {"claim":"Structured messages improve handoff traceability.","source":"demo-note-B","confidence":.88},
 {"claim":"Unbounded agent loops can increase execution cost.","source":"demo-note-C","confidence":.95},
 {"claim":"Human review is important before consequential actions.","source":"demo-note-D","confidence":.97}]}),
"T2":Task("T2","Synthesize findings","analyst",["T1"]),
"T3":Task("T3","Draft brief","writer",["T2"]),
"T4":Task("T4","Review deliverable","reviewer",["T3"])}
def topological_order(task_map):
    degree={k:0 for k in task_map}; graph=defaultdict(list)
    for k,t in task_map.items():
        for dep in t.dependencies:
            if dep not in task_map: raise ValueError(f"Missing dependency {dep}")
            graph[dep].append(k); degree[k]+=1
    q=deque(k for k,v in degree.items() if v==0); order=[]
    while q:
        k=q.popleft(); order.append(k)
        for nxt in graph[k]:
            degree[nxt]-=1
            if degree[nxt]==0:q.append(nxt)
    if len(order)!=len(task_map): raise ValueError("Dependency cycle detected")
    return order
order=topological_order(tasks); print("Execution order:",order)


## 5. Execute agents with bounded retries

In [ ]:
def execute(task_map,max_attempts=2):
    for tid in topological_order(task_map):
        t=task_map[tid]
        if not all(task_map[d].status=="completed" for d in t.dependencies):
            t.status="blocked"; continue
        t.status="running"
        while t.attempts<max_attempts:
            t.attempts+=1
            try:
                t.output=agents[t.owner].run(t); t.status="completed"; break
            except Exception as e:
                t.output={"error":str(e)}; t.status="failed"
        if t.status!="completed": break
    return task_map
tasks=execute(tasks)
display(pd.DataFrame([{"task":t.task_id,"owner":t.owner,"status":t.status,"attempts":t.attempts} for t in tasks.values()]))


## 6. Inspect deliverable and review

In [ ]:
print(workspace.artifacts["draft"]["draft"])
print("\nReview:",json.dumps(workspace.artifacts["review"],indent=2))


## 7. Inspect agent messages

In [ ]:
messages=pd.DataFrame([asdict(m) for m in bus.messages])
display(messages[["message_id","sender","recipient","task_id","kind","payload"]])


## 8. Visualize collaboration

In [ ]:
if not messages.empty:
    messages.groupby("sender").size().plot(kind="bar",figsize=(7,3),title="Messages sent by agent")
    plt.ylabel("Message count"); plt.tight_layout(); plt.show()
pd.Series([t.status for t in tasks.values()]).value_counts().plot(kind="bar",title="Task status")
plt.ylabel("Task count"); plt.tight_layout(); plt.show()


## 9. Export artifacts and audit data

In [ ]:
out="agent_collaboration_outputs"; os.makedirs(out,exist_ok=True)
open(os.path.join(out,"collaboration_brief.md"),"w",encoding="utf-8").write(workspace.artifacts["draft"]["draft"])
with open(os.path.join(out,"workflow_report.json"),"w",encoding="utf-8") as f:
    json.dump({"tasks":{k:{"status":t.status,"attempts":t.attempts} for k,t in tasks.items()},
               "review":workspace.artifacts["review"],"messages":[asdict(m) for m in bus.messages],
               "audit":workspace.audit},f,indent=2)
messages.to_csv(os.path.join(out,"agent_messages.csv"),index=False)
print("Created:",os.listdir(out))


## 10. Production extensions and limitations

**Extensions:** LLM-backed agents with structured outputs; LangGraph, AutoGen, or CrewAI adapters; persistent event storage; asynchronous workers; tool allowlists and sandboxing; per-agent permissions; correlation IDs and idempotency; observability and cost tracking; human approval gates.

**Limitations:** This is a deterministic local simulation, not a distributed multi-agent service. It does not call an LLM or external tools. The reviewer checks document structure, not factual accuracy. Supplied evidence is illustrative. Require human review before consequential actions.